<a href="https://colab.research.google.com/github/ahmadkiyani2452/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# My Rule

The goal of this baseline rule is to identify pages that are good candidates for content refresh.

The rule gives a higher score to pages that:

- have not been updated for a long time,
- have a low organic CTR,
- have good search demand.

These signals suggest that the content may be outdated or underperforming and could benefit from a refresh.

## Signals Checked

### Signal 1
Days Since Update

Verdict: CONFIRMED

Older pages are more likely to require content updates.

### Signal 2
Organic CTR

Verdict: CONFIRMED

Pages with low CTR despite receiving impressions may benefit from improved titles, descriptions, or refreshed content.

## Reason Codes

- STALE_CONTENT
- LOW_CTR
- HIGH_SEARCH_VOLUME

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
!git clone https://github.com/ahmadkiyani2452/flyrank-internship.git

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 178 (delta 60), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 8.14 MiB | 11.55 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [ ]:
%cd /content/flyrank-internship

/content/flyrank-internship


In [ ]:
import os

print(os.getcwd())
print(os.listdir())

/content/flyrank-internship
['notebooks', 'work', 'AGENTS.md', 'skills', 'DATA_USE.md', 'scripts', '01_first_look_and_discovery.ipynb', '.github', 'LICENSE', '.gitignore', 'outputs', 'CLAUDE.md', 'data', 'README.md', 'submission', '.git', 'GUIDE.md', 'SETUP.md', 'requirements.txt', 'docs']


In [ ]:
os.listdir("work")

['capstone_report_template.md',
 'notebooks',
 'curated-images',
 'identity-kit',
 'README.md']

In [ ]:
os.listdir("work/notebooks")

['w01_research_question.ipynb',
 'w06_validation_audit.ipynb',
 'w07_action_playbook.ipynb',
 'w04_baseline_score.ipynb',
 'w05_model.ipynb',
 'w04_signal_audit.ipynb',
 'w03_feature_leakage_check.ipynb',
 'w02_ml_task_framing.ipynb',
 'w03_data_contract.ipynb',
 'capstone.ipynb']

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [ ]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
import pandas as pd

age_table = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_search_volume=("search_volume", "mean")
      )
)

print("Signal 1: Freshness")
display(age_table)

Signal 1: Freshness


,n,avg_ctr,avg_search_volume
freshness_tier,,,
0-30,20480,0.609021,158.441911
181+,174,3.693276,38.000000
31-90,175,0.117543,788.448276
91-180,9171,0.238367,148.585790


In [ ]:
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[0,0.5,1,2,5,100],
    labels=["0-0.5","0.5-1","1-2","2-5","5+"]
)

ctr_table = (
    df.groupby("ctr_bucket")
      .agg(
          n=("content_id","count"),
          avg_position=("avg_position","mean"),
          avg_search_volume=("search_volume","mean")
      )
)

print("Signal 2: CTR")
display(ctr_table)

Signal 2: CTR


/tmp/ipykernel_1799/1964744350.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("ctr_bucket")


,n,avg_position,avg_search_volume
ctr_bucket,,,
0-0.5,12639,15.281027,136.073008
0.5-1,2460,11.489797,61.409987
1-2,915,11.658142,82.069378
2-5,333,10.507508,101.885246
5+,441,7.869615,282.297297


In [ ]:
score = pd.Series(0, index=df.index)

# Old content
score += (df["days_since_last_update"] > 180) * 40

# Low CTR
score += (df["ctr"] < 1) * 30

# High Search Volume
score += (df["search_volume"] >= 100) * 30

df["baseline_score"] = score

In [ ]:
def reason(row):

    reasons=[]

    if row["days_since_last_update"]>180:
        reasons.append("STALE_CONTENT")

    if row["ctr"]<1:
        reasons.append("LOW_CTR")

    if row["search_volume"]>=100:
        reasons.append("HIGH_SEARCH_VOLUME")

    return "|".join(reasons)

df["reason_code"]=df.apply(reason,axis=1)

In [ ]:
df["action"]="No Action"

df.loc[df["baseline_score"]>=70,"action"]="Refresh Content"

df.loc[(df["baseline_score"]>=30) & (df["baseline_score"]<70),"action"]="Monitor"

In [ ]:
ranked = df.sort_values(
    by="baseline_score",
    ascending=False
)

ranked.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_bucket,baseline_score,reason_code,action
15947,content_40e140ba2934,client_8722616204,720.0,0.29,LOW,0.78,keyword article,transactional,3306.0,21228.0,...,5.56,0.0,low,page_1,flat,NaN,NaN,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,...,0.00,0.0,low,striking,down,-100.0,NaN,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,...,50.00,0.0,low,top_3,down,-100.0,NaN,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.00,0.0,low,page_1,down,-95.6,NaN,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
6103,content_691fc6b910e6,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6426.0,42872.0,...,0.00,0.0,low,striking,stable,-12.0,NaN,70,STALE_CONTENT|LOW_CTR,Refresh Content


In [ ]:
import os

os.makedirs("work/outputs",exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked.head(20)

top20[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]]

,content_id,baseline_score,reason_code,action
15947,content_40e140ba2934,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
1659,content_bbca724138f2,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
23619,content_24abafed9707,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
21984,content_02b0d6e30129,100,STALE_CONTENT|LOW_CTR|HIGH_SEARCH_VOLUME,Refresh Content
6103,content_691fc6b910e6,70,STALE_CONTENT|LOW_CTR,Refresh Content
4013,content_3b7c76f80c79,70,STALE_CONTENT|LOW_CTR,Refresh Content
16475,content_f2b4acf220d9,70,STALE_CONTENT|LOW_CTR,Refresh Content
29364,content_46d354489e12,70,STALE_CONTENT|LOW_CTR,Refresh Content
6119,content_cd27391ecd03,70,STALE_CONTENT|LOW_CTR,Refresh Content
7222,content_8f2c815af658,70,STALE_CONTENT|LOW_CTR,Refresh Content


### Top-20 Review

Most of the highest-ranked pages received the **Refresh Content** action because they combine stale content, low CTR, and good search demand.

Confidence: Medium

These recommendations are based only on observable content signals.

What would make them wrong?

- Missing or inaccurate last update dates.
- Seasonal traffic changes.
- Temporary ranking fluctuations.
- Search volume estimation errors.
- Content already refreshed but metadata not updated.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive high scores because of incomplete metadata or temporary performance changes rather than genuine refresh needs.

Examples include:

- Seasonal content
- Recently updated pages with outdated timestamps
- Pages affected by temporary Google ranking changes

## Leakage Check

No future information was used.

The rule only uses:

- Days Since Last Update
- CTR
- Search Volume

No labels, future performance, or product flags were included.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.